In [0]:
# Databricks notebook source
# PATH: nyc-taxi-pipeline/notebooks/02_silver_clean.py

from pyspark.sql.functions import col, unix_timestamp
from delta.tables import DeltaTable

# 1. Define input parameters/widgets for Airflow integration
dbutils.widgets.text("year", "2026")
dbutils.widgets.text("month", "01")
year = dbutils.widgets.get("year")
month = f"{int(dbutils.widgets.get('month')):02d}"

# 2. Optimized Fetch: Push filters directly into the read operation
print(f"Reading and filtering Bronze batch for {year}-{month}...")
df_bronze_batch = spark.read.table("nyc_taxi_medallion.bronze_yellow_trips") \
    .filter((col("source_year") == year) & (col("source_month") == month))

# 3. Apply operational transformations and cleaning rules
print("Applying cleaning parameters and calculating trip durations...")
df_silver_batch = df_bronze_batch.filter(
    (col("trip_distance") > 0) & 
    (col("passenger_count") > 0) & 
    (col("total_amount") > 0)
).withColumn(
    "trip_duration_minutes", 
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60
).filter(
    (col("trip_duration_minutes") > 0) & (col("trip_duration_minutes") < 180)
)

# 4. Handle Save/Merge Layout
target_table_name = "nyc_taxi_medallion.silver_yellow_trips"

if not spark.catalog.tableExists(target_table_name):
    print(f"Creating initialization schema for target: {target_table_name}")
    # Optimization: Partition the Silver table by year/month just like Bronze for fast lookups
    df_silver_batch.write \
        .format("delta") \
        .partitionBy("source_year", "source_month") \
        .saveAsTable(target_table_name)
    print("Silver table created and base data loaded!")
else:
    print(f"Target table found. Executing partition-pruned Delta merge...")
    target_table = DeltaTable.forName(spark, target_table_name)
    
    # Optimization: Add partition pruners into the merge condition. 
    # This instructs Delta to completely ignore all historical partitions and focus only on the active target block.
    optimized_merge_condition = f"""
        target.source_year = '{year}' AND 
        target.source_month = '{month}' AND 
        target.VendorID = updates.VendorID AND 
        target.tpep_pickup_datetime = updates.tpep_pickup_datetime
    """
    
    target_table.alias("target").merge(
        source = df_silver_batch.alias("updates"),
        condition = optimized_merge_condition
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
     
    print(f"Incremental batch for {year}-{month} merged into Silver successfully!")